# R Master v8a Fix1 · Lap Head Graft · Low-Memory Checkpoint

这是 v8a 的 **断线修复版**。两次都出现 Colab 从运行状态变成“正在连接”，按两次阻断规则不再重跑原方案。

### 改动
原 v8a 会在同一个 Blender 进程里同时打开：
- v7c Mona 完整工程
- Lapine 约 102 MB FBX

这会制造明显的峰值内存和瞬时 CPU/IO 压力。Fix1 改成两段：

1. **Donor Extract**：空 Blender 只导入 Lapine，提取头部 mesh，立刻保存一个轻量 donor checkpoint 到 Drive。
2. **Graft Build**：关闭第一进程后，另开 Blender 只加载 v7c，再 append 轻量 donor。

即使 Colab/手机前端掉线，完成的 checkpoint 会保留，重跑自动跳过。

### 不变
- Mona 505 骨不改
- 不烘焙 Rest Pose
- 不导 VRM
- 不写 Lapine 原始资产到 GitHub
- 只做接头预览


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, zipfile, tarfile, json, os, hashlib, time

BUILD_TAG="v8a_fix1_lowmem_20260919_r1"
print("R Master v8a Fix1 · Low-Memory Checkpoint")
drive.mount("/content/drive")

ROOT=Path("/content/drive/MyDrive/R_Master")
SRC=ROOT/"v7c_refine"/"latest"/"R_Master_v7c_REFINED_PREVIEW.blend"
REF=ROOT/"reference"
CACHE=ROOT/"cache"
CP=ROOT/"v8a_head_graft"/"checkpoint"
OUT=ROOT/"v8a_head_graft"/"fix1_latest"
for p in (REF,CACHE,CP,OUT): p.mkdir(parents=True,exist_ok=True)

if not SRC.exists() or SRC.stat().st_size < 50*1024*1024:
    raise RuntimeError("未找到 v7c 预览文件。把这一行截图给二蛋。")
print(f"✓ v7c：{SRC.stat().st_size/1024/1024:.1f} MiB")

LAPZIP=REF/"Lapine_Ver.1.11_A.zip"
if not LAPZIP.exists() or LAPZIP.stat().st_size < 70*1024*1024:
    print("Drive 里还没有 Lapine 缓存。请选择你自己的 Lapine_Ver.1.11_A.zip（只需这一次）。")
    up=files.upload()
    if not up: raise RuntimeError("未选择 Lapine zip")
    name=next(iter(up))
    tmp=Path("/content")/name
    tmp.write_bytes(up[name])
    shutil.copy2(tmp,LAPZIP)
    print("✓ Lapine 已缓存到 Drive")
else:
    print("✓ 已找到 Lapine Drive 缓存，不再上传")

h=hashlib.sha256()
with LAPZIP.open("rb") as f:
    for b in iter(lambda:f.read(8*1024*1024),b""): h.update(b)
sha=h.hexdigest()
EXPECTED="8e215aeae8d2e42719305d66292b84c2714f616b7952591dee0383c376c86af4"
print("Lapine SHA256:",sha)
if sha!=EXPECTED: raise RuntimeError("Lapine 源校验不一致，停止")
print("✓ Lapine 源校验通过")


In [ ]:
BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
LOCAL=Path("/content/r_master_v8a_fix1"); LOCAL.mkdir(parents=True,exist_ok=True)
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size>100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE); print("✓ 复用 Blender 缓存")
else:
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)

if not (BDIR/"blender").exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)
BLENDER=BDIR/"blender"
print("✓ Blender 就绪")

# 只从 unitypackage 临时提取 Lapine.fbx
UNPACK=LOCAL/"lapine_unpack"
if UNPACK.exists(): shutil.rmtree(UNPACK)
UNPACK.mkdir(parents=True)
outer=UNPACK/"outer"
with zipfile.ZipFile(LAPZIP,"r") as z:
    unity=[n for n in z.namelist() if n.lower().endswith("lapine.unitypackage")]
    if not unity: raise RuntimeError("zip 内无 Lapine.unitypackage")
    z.extract(unity[0],outer)
UNITY=outer/unity[0]
pkg=UNPACK/"pkg"; pkg.mkdir()
with tarfile.open(UNITY,"r:*") as t: t.extractall(pkg)

FBX=None
for p in pkg.glob("*/pathname"):
    try: q=p.read_text(encoding="utf-8").strip()
    except: continue
    if q=="Assets/Models/FBX/Lapine.fbx":
        a=p.parent/"asset"
        if a.exists():
            FBX=UNPACK/"Lapine.fbx"; shutil.copy2(a,FBX); break
if FBX is None: raise RuntimeError("没找到 Lapine.fbx")
print(f"✓ 临时 Lapine.fbx：{FBX.stat().st_size/1024/1024:.1f} MiB")


In [ ]:
DONOR_BLEND=CP/"Lapine_Head_Donor_v8a_fix1.blend"
DONOR_META=CP/"Lapine_Head_Donor_v8a_fix1.json"
DONOR_SCRIPT=LOCAL/"extract_donor.py"
DONOR_SCRIPT.write_text("\nimport bpy,sys,os,json\nfrom mathutils import Vector\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nfbx=out=meta=None\nfor i,a in enumerate(argv):\n    if a==\"--fbx\": fbx=argv[i+1]\n    if a==\"--out\": out=argv[i+1]\n    if a==\"--meta\": meta=argv[i+1]\nif not fbx or not out or not meta: raise RuntimeError(\"args\")\n\nbpy.ops.wm.read_factory_settings(use_empty=True)\nbpy.ops.import_scene.fbx(filepath=fbx,automatic_bone_orientation=False)\n\nrigs=[o for o in bpy.data.objects if o.type==\"ARMATURE\"]\nif not rigs: raise RuntimeError(\"Lapine armature missing\")\nrig=max(rigs,key=lambda o:len(o.data.bones))\n\ndef find_bone(names):\n    for n in names:\n        b=rig.pose.bones.get(n)\n        if b:return b\n    for b in rig.pose.bones:\n        ln=b.name.lower()\n        if any(ln.endswith(n.lower()) for n in names): return b\n    return None\n\nhb=find_bone([\"Head\",\"head\"])\nnb=find_bone([\"Neck\",\"neck\"])\nif not hb or not nb: raise RuntimeError(\"Lapine Head/Neck missing\")\nh=rig.matrix_world@hb.head\nn=rig.matrix_world@nb.head\n\ntokens=(\"face\",\"eye\",\"brow\",\"hair\",\"lash\",\"mouth\",\"teeth\",\"tooth\",\"tongue\",\"jaw\",\"ear\")\nmeshes=[o for o in bpy.data.objects if o.type==\"MESH\"]\nsel=[]\nfor o in meshes:\n    if any(t in o.name.lower() for t in tokens):\n        sel.append(o)\n\n# 空间兜底：绝大部分 bbox 位于 neck 以上的 mesh 视为头部组件。\nif len(sel)<3:\n    for o in meshes:\n        if o in sel: continue\n        pts=[o.matrix_world@Vector(c) for c in o.bound_box]\n        if pts and sum(p.z>n.z for p in pts)>=6:\n            sel.append(o)\nif not sel: raise RuntimeError(\"0 Lapine head meshes\")\n\n# 把 donor mesh 固化为静态预览几何，去掉 armature 依赖。\nfor o in sel:\n    for m in list(o.modifiers):\n        if m.type==\"ARMATURE\": o.modifiers.remove(m)\n    o[\"R_DONOR\"]=\"LapineHead_v8a_fix1\"\n\nfor o in list(bpy.data.objects):\n    if o not in sel:\n        bpy.data.objects.remove(o,do_unlink=True)\n\nbpy.ops.wm.save_as_mainfile(filepath=out,check_existing=False)\nwith open(meta,\"w\",encoding=\"utf-8\") as f:\n    json.dump({\"head_world\":list(map(float,h)),\"neck_world\":list(map(float,n)),\n               \"meshes\":[o.name for o in sel],\"mesh_count\":len(sel)},f,ensure_ascii=False,indent=2)\nprint(\"[v8a Fix1] DONOR_OK\",len(sel))\n",encoding="utf-8")

need=True
if DONOR_BLEND.exists() and DONOR_BLEND.stat().st_size>2*1024*1024 and DONOR_META.exists():
    try: need=json.loads(DONOR_META.read_text(encoding="utf-8")).get("mesh_count",0)<1
    except: need=True

if need:
    print("① 提取 Lap 头 donor（这一段完成后会永久 checkpoint）")
    log=CP/"donor_extract.log"
    cmd=["xvfb-run","-a",str(BLENDER),"--background","--factory-startup","--python",str(DONOR_SCRIPT),
         "--","--fbx",str(FBX),"--out",str(DONOR_BLEND),"--meta",str(DONOR_META)]
    with log.open("w",encoding="utf-8") as f:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            f.write(line)
            if "[v8a Fix1]" in line or "Traceback" in line or "Error" in line: print(line.rstrip())
        rc=p.wait()
    if rc!=0:
        print(log.read_text(encoding="utf-8",errors="replace")[-12000:])
        raise RuntimeError("donor 提取失败")
else:
    print("✓ donor checkpoint 已存在，跳过 Lapine 大 FBX 导入")
print(json.loads(DONOR_META.read_text(encoding="utf-8")))


In [ ]:
GRAFT_SCRIPT=LOCAL/"graft_fix1.py"
GRAFT_SCRIPT.write_text("\nimport bpy,sys,os,json\nfrom mathutils import Vector,Matrix\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\ndonor=meta=out=tag=None\nfor i,a in enumerate(argv):\n    if a==\"--donor\": donor=argv[i+1]\n    if a==\"--meta\": meta=argv[i+1]\n    if a==\"--out\": out=argv[i+1]\n    if a==\"--tag\": tag=argv[i+1]\nif not donor or not meta or not out: raise RuntimeError(\"args\")\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nrig=bpy.data.objects.get(\"R_Master_Align_v2_PREVIEW\") or bpy.data.objects.get(\"Mona_Armature\")\nif not body or not rig: raise RuntimeError(\"Mona source objects missing\")\n\ndef bone_any(r,names):\n    for n in names:\n        b=r.pose.bones.get(n)\n        if b:return b\n    for b in r.pose.bones:\n        if any(b.name.lower().endswith(n.lower()) for n in names): return b\n    return None\n\nmh=bone_any(rig,[\"Head\",\"head\"])\nmn=bone_any(rig,[\"Neck2\",\"Neck1\",\"Neck\",\"neck\"])\nif not mh or not mn: raise RuntimeError(\"Mona Head/Neck missing\")\nm_head=rig.matrix_world@mh.head\nm_neck=rig.matrix_world@mn.head\n\ndm=json.load(open(meta,\"r\",encoding=\"utf-8\"))\nl_head=Vector(dm[\"head_world\"]); l_neck=Vector(dm[\"neck_world\"])\nlv=l_head-l_neck; mv=m_head-m_neck\nscale=mv.length/max(1e-8,lv.length)\nrot=lv.normalized().rotation_difference(mv.normalized())\nX=Matrix.Translation(m_neck) @ rot.to_matrix().to_4x4() @ Matrix.Scale(scale,4) @ Matrix.Translation(-l_neck)\n\nbefore=set(bpy.data.objects)\nwith bpy.data.libraries.load(donor,link=False) as (src,dst):\n    dst.objects=list(src.objects)\nfor o in dst.objects:\n    if o is not None: bpy.context.collection.objects.link(o)\ndonors=[o for o in bpy.data.objects if o not in before and o.type==\"MESH\"]\nif not donors: raise RuntimeError(\"append donor 0 mesh\")\nfor o in donors:\n    o.matrix_world=X@o.matrix_world\n    o[\"R_DONOR\"]=\"LapineHead_v8a_fix1\"\n\n# Mona head deletion on preview duplicate.\ngroup_ids=[body.vertex_groups[n].index for n in (\"Head\",\"HeadStretch\") if n in body.vertex_groups]\ntod=[]\nmw=body.matrix_world.copy()\nfor v in body.data.vertices:\n    p=mw@v.co\n    weights={g.group:g.weight for g in v.groups}\n    hw=max([weights.get(i,0.0) for i in group_ids] or [0.0])\n    if p.z>m_neck.z+0.025 and hw>0.08: tod.append(v.index)\nif not tod: raise RuntimeError(\"Mona head remove 0 verts\")\nbpy.context.view_layer.objects.active=body; body.select_set(True)\nbpy.ops.object.mode_set(mode=\"EDIT\"); bpy.ops.mesh.select_all(action=\"DESELECT\")\nbpy.ops.object.mode_set(mode=\"OBJECT\")\nfor i in tod: body.data.vertices[i].select=True\nbpy.ops.object.mode_set(mode=\"EDIT\"); bpy.ops.mesh.delete(type=\"VERT\")\nbpy.ops.object.mode_set(mode=\"OBJECT\"); body.select_set(False)\n\n# 仅保留 body + donor meshes；armature 可留但渲染隐藏。\nfor o in list(bpy.data.objects):\n    if o.type==\"MESH\" and o!=body and o not in donors:\n        bpy.data.objects.remove(o,do_unlink=True)\n\nrep={\"ok\":True,\"stage\":\"R_Master_v8a_Fix1_LowMemoryHeadGraft\",\"build_tag\":tag,\n     \"mona_bone_count\":len(rig.data.bones),\"donor_mesh_count\":len(donors),\n     \"donor_meshes\":[o.name for o in donors],\"mona_head_vertices_removed\":len(tod),\n     \"uniform_scale\":float(scale),\"preview_only\":True,\"weights_transferred\":False,\n     \"neck_welded\":False,\"rest_pose_baked\":False,\"final_vrm\":False}\nwith open(os.path.join(out,\"R_Master_v8a_Fix1_report.json\"),\"w\",encoding=\"utf-8\") as f:\n    json.dump(rep,f,ensure_ascii=False,indent=2)\nbpy.ops.wm.save_as_mainfile(filepath=os.path.join(out,\"R_Master_v8a_Fix1_HEAD_GRAFT_PREVIEW.blend\"),check_existing=False)\nprint(\"[v8a Fix1] GRAFT_OK\",len(donors),len(tod),scale)\n",encoding="utf-8")
STAGE=OUT/"R_Master_v8a_Fix1_HEAD_GRAFT_PREVIEW.blend"
REPORT=OUT/"R_Master_v8a_Fix1_report.json"
need=True
if STAGE.exists() and STAGE.stat().st_size>50*1024*1024 and REPORT.exists():
    try: need=json.loads(REPORT.read_text(encoding="utf-8")).get("build_tag")!=BUILD_TAG
    except: need=True
if need:
    print("② 加载 v7c + 轻量 donor，开始接头")
    TMP=LOCAL/"graft"; 
    if TMP.exists(): shutil.rmtree(TMP)
    TMP.mkdir()
    log=TMP/"graft.log"
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(SRC),"--python",str(GRAFT_SCRIPT),
         "--","--donor",str(DONOR_BLEND),"--meta",str(DONOR_META),"--out",str(TMP),"--tag",BUILD_TAG]
    with log.open("w",encoding="utf-8") as f:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            f.write(line)
            if "[v8a Fix1]" in line or "Traceback" in line or "Error" in line: print(line.rstrip())
        rc=p.wait()
    if rc!=0:
        print(log.read_text(encoding="utf-8",errors="replace")[-12000:])
        raise RuntimeError("接头构建失败")
    for n in ("R_Master_v8a_Fix1_HEAD_GRAFT_PREVIEW.blend","R_Master_v8a_Fix1_report.json"):
        shutil.copy2(TMP/n,OUT/n)
    shutil.copy2(log,OUT/"R_Master_v8a_Fix1_graft.log")
else:
    print("✓ 接头 checkpoint 已存在，跳过")
print(json.loads(REPORT.read_text(encoding="utf-8")))


In [ ]:
RENDER_SCRIPT=LOCAL/"render_fix1.py"
RENDER_SCRIPT.write_text("\nimport bpy,os,sys\nfrom mathutils import Vector\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=view=None\nfor i,a in enumerate(argv):\n    if a==\"--out\": out=argv[i+1]\n    if a==\"--view\": view=argv[i+1]\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\ndonors=[o for o in bpy.data.objects if o.type==\"MESH\" and o.get(\"R_DONOR\")==\"LapineHead_v8a_fix1\"]\nif not body or not donors: raise RuntimeError(\"objects missing\")\nfor o in bpy.context.scene.objects:\n    if o.type==\"ARMATURE\": o.hide_render=True\n    if o.type==\"MESH\": o.hide_render=(o!=body and o not in donors)\npts=[]\nfor o in [body]+donors: pts += [o.matrix_world@Vector(c) for c in o.bound_box]\nmn=Vector((min(p.x for p in pts),min(p.y for p in pts),min(p.z for p in pts)))\nmx=Vector((max(p.x for p in pts),max(p.y for p in pts),max(p.z for p in pts)))\nc=(mn+mx)*.5; h=mx.z-mn.z; d=max(h,mx.x-mn.x,mx.y-mn.y)*2.6\ns=bpy.context.scene; s.render.engine=\"BLENDER_WORKBENCH\"; s.render.image_settings.file_format=\"PNG\"\ns.display.shading.light=\"STUDIO\"; s.display.shading.show_shadows=True; s.display.shading.show_cavity=True\ns.display.shading.cavity_type=\"WORLD\"; s.display.shading.color_type=\"SINGLE\"; s.display.shading.single_color=(.62,.62,.65)\ns.display.shading.background_type=\"VIEWPORT\"; s.display.shading.background_color=(.04,.04,.05)\ncd=bpy.data.cameras.new(\"R_v8a_fix1_cam_data\"); cam=bpy.data.objects.new(\"R_v8a_fix1_cam\",cd); s.collection.objects.link(cam); s.camera=cam\ncam.data.type=\"ORTHO\"\ndef look(t): cam.rotation_euler=(Vector(t)-cam.location).to_track_quat(\"-Z\",\"Y\").to_euler()\ndef rr(fn,pos,tgt,scale,res):\n    s.render.resolution_x,s.render.resolution_y=res; s.render.resolution_percentage=100\n    cam.location=Vector(pos); cam.data.ortho_scale=scale; look(tgt); s.render.filepath=os.path.join(out,fn); bpy.ops.render.render(write_still=True)\nheadz=mx.z-h*.10; neckz=mx.z-h*.20\nspec={\n\"front\":(\"R_Master_v8a_Fix1_front.png\",(c.x,c.y-d,c.z),c,h*1.08,(640,900)),\n\"side\":(\"R_Master_v8a_Fix1_side.png\",(c.x+d,c.y,c.z),c,h*1.08,(640,900)),\n\"back\":(\"R_Master_v8a_Fix1_back.png\",(c.x,c.y+d,c.z),c,h*1.08,(640,900)),\n\"three_quarter\":(\"R_Master_v8a_Fix1_three_quarter.png\",(c.x+d*.72,c.y-d*.72,c.z),c,h*1.08,(640,900)),\n\"head_front\":(\"R_Master_v8a_Fix1_head_front.png\",(c.x,c.y-d,headz),(c.x,c.y,headz),h*.30,(800,800)),\n\"head_side\":(\"R_Master_v8a_Fix1_head_side.png\",(c.x+d,c.y,headz),(c.x,c.y,headz),h*.30,(800,800)),\n\"head_three_quarter\":(\"R_Master_v8a_Fix1_head_three_quarter.png\",(c.x+d*.72,c.y-d*.72,headz),(c.x,c.y,headz),h*.30,(800,800)),\n\"neck_interface\":(\"R_Master_v8a_Fix1_neck_interface.png\",(c.x+d*.70,c.y-d*.70,neckz),(c.x,c.y,neckz),h*.24,(800,650))}\nrr(*spec[view])\nprint(\"[v8a Fix1] RENDER_OK\",view)\n",encoding="utf-8")
views=[
("front","R_Master_v8a_Fix1_front.png"),("side","R_Master_v8a_Fix1_side.png"),
("back","R_Master_v8a_Fix1_back.png"),("three_quarter","R_Master_v8a_Fix1_three_quarter.png"),
("head_front","R_Master_v8a_Fix1_head_front.png"),("head_side","R_Master_v8a_Fix1_head_side.png"),
("head_three_quarter","R_Master_v8a_Fix1_head_three_quarter.png"),("neck_interface","R_Master_v8a_Fix1_neck_interface.png")]
for i,(v,f) in enumerate(views,1):
    p=OUT/f
    if p.exists() and p.stat().st_size>15000:
        print(f"✓ [{i}/8] {v} checkpoint"); continue
    print(f"③ [{i}/8] 渲染 {v}")
    r=subprocess.run(["xvfb-run","-a",str(BLENDER),"--background",str(STAGE),"--python",str(RENDER_SCRIPT),
                      "--","--out",str(OUT),"--view",v],
                     stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if r.returncode!=0:
        print(r.stdout[-10000:]); raise RuntimeError(v+" 渲染失败")
print("✓ 8 视图全部完成")


In [ ]:
from IPython.display import display,Image,Markdown
items=[
("全身正面","R_Master_v8a_Fix1_front.png"),("全身侧面","R_Master_v8a_Fix1_side.png"),
("全身背面","R_Master_v8a_Fix1_back.png"),("全身 3/4","R_Master_v8a_Fix1_three_quarter.png"),
("头肩正面","R_Master_v8a_Fix1_head_front.png"),("头肩侧面","R_Master_v8a_Fix1_head_side.png"),
("头肩 3/4","R_Master_v8a_Fix1_head_three_quarter.png"),("颈部接口","R_Master_v8a_Fix1_neck_interface.png")]
for title,f in items:
    display(Markdown("### "+title)); display(Image(filename=str(OUT/f),width=480))
z=OUT/"R_Master_v8a_Fix1_Review.zip"
if z.exists(): z.unlink()
with zipfile.ZipFile(z,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as w:
    for _,f in items: w.write(OUT/f,arcname=f)
    for f in ("R_Master_v8a_Fix1_report.json","R_Master_v8a_Fix1_graft.log"):
        if (OUT/f).exists(): w.write(OUT/f,arcname=f)
print(f"✓ Review ZIP：{z.stat().st_size/1024/1024:.1f} MiB")
files.download(str(z))
